In [13]:
import torch
import numpy as np
# https://blog.csdn.net/bit452/article/details/109686936

- softmax的输入不需要再做非线性变换，也就是说softmax之前不再需要激活函数(relu)。softmax两个作用，如果在进行softmax前的input有负数，通过指数变换，得到正数。所有类的概率求和为1。<br>
- y的标签编码方式是one-hot。我对one-hot的理解是只有一位是1，其他位为0。(但是标签的one-hot编码是算法完成的，算法的输入仍为原始标签)<br>
- 多分类问题，标签y的类型是LongTensor。比如说0-9分类问题，如果y = torch.LongTensor([3])，对应的one-hot是[0,0,0,1,0,0,0,0,0,0].(这里要注意，如果使用了one-hot，标签y的类型是LongTensor，糖尿病数据集中的target的类型是FloatTensor)<br>
- CrossEntropyLoss <==> LogSoftmax + NLLLoss。也就是说使用CrossEntropyLoss最后一层(线性层)是不需要做其他变化的；使用NLLLoss之前，需要对最后一层(线性层)先进行SoftMax处理，再进行log操作。      

In [14]:
# 用numpy实现 softmax激活函数
y = np.array([ 1, 0, 0])
z = np.array([ 0.2, 0.1, -0.1])
y_pred = np.exp(z) / np.exp(z).sum()
loss = (-y * np.log(y_pred)).sum()
print(loss)

0.9729189131256584


In [15]:
import torch
# 用torch实现 softmax激活函数
y = torch.LongTensor([0])
z = torch.Tensor([ [0.2, 0.1, -0.1]])
criterion = torch.nn.CrossEntropyLoss() # 交叉熵损失
loss = criterion(z,y)
print(loss)


tensor(0.9729)


In [16]:
import torch
criterion = torch.nn.CrossEntropyLoss()
Y = torch.LongTensor([2, 0, 1])
Y_pred1 = torch.Tensor([
    [0.1 , 0.2, 0.9],
    [1.1, 0.1, 0.2],
    [0.2, 2.1, 0.1]
])
Y_pred2 = torch.Tensor([
    [0.8, 0.2, 0.3],
    [0.2, 0.3, 0.5],
    [0.2, 0.2, 0.5]
])
loss1 = criterion(Y_pred1, Y)
loss2 = criterion(Y_pred2, Y)
print("Batch Loss1 = ", loss1.data, "\nBatch Loss2 = ", loss2.data)

Batch Loss1 =  tensor(0.4966) 
Batch Loss2 =  tensor(1.2389)


In [17]:
import torch
from torchvision import transforms
from torchvision import datasets
from torch.utils.data import DataLoader
import torch.nn.functional as F 
import torch. optim as optim 

# prepare dataset
batch_size = 64
transform = transforms.Compose([
    transforms.ToTensor(), # 将PIL或者opencv的 W * H * C 张量 转为 C * W * H
    transforms.Normalize( (0.1307,), (0.3081, )) # 进行归一化处理
])
train_dataset = datasets.MNIST(root = "../dataset/mnist/",
                                train = True,
                                download=True,
                                transform=transform
                               )
train_loader = DataLoader(train_dataset,
                          shuffle=True,
                          batch_size=batch_size
                          )
test_dataset = datasets.MNIST(root = "../dataset/mnist/",
                              train=False,
                              download=True,
                              transform=transform
                              )
test_loader =DataLoader(test_dataset,
                        shuffle=False,
                        batch_size=batch_size
                        )

# design model using class
class Net(torch.nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.linear1 = torch.nn.Linear(784, 512)
        self.linear2 = torch.nn.Linear(512, 256)
        self.linear3 = torch.nn.Linear(256, 128)
        self.linear4 = torch.nn.Linear(128, 64)
        self.linear5 = torch.nn.Linear(64, 10)

    def forward(self, x):
        x = x.view(-1, 784)
        x = F.relu(self.linear1(x))
        x = F.relu(self.linear2(x))
        x = F.relu(self.linear3(x))
        x = F.relu(self.linear4(x))
        return self.linear5(x)
    
model = Net()

# construct loss and optimizer
criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr = 0.01, momentum=0.5)

# training cycle
def train(epoch):
    running_loss = 0.0
    for batch_idx, data in enumerate(train_loader, 0):
        inputs, target = data
        optimizer.zero_grad()

        # Forward
        outputs = model(inputs)
        # Loss
        loss = criterion(outputs, target)
        # Backward
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        if batch_idx % 300 == 299:
            print("[ %d, %5d] loss: %.3f" % (epoch + 1, batch_idx + 1, running_loss / 300));

def test():
    correct = 0
    total = 0
    with torch.no_grad(): # https://blog.csdn.net/ego_bai/article/details/80873242
        for data in test_loader:
            images, labels = data
            outputs = model(images)
            _, predicted = torch.max(outputs.data, dim = 1) # dim = 0 列是第零个维度，行是一个维度  url:https://blog.csdn.net/qq_40210586/article/details/103874000
            ''' 
            在代码中，下划线 "_" 是一个惯例，通常用作一个临时变量名，用于表示某个值或结果我们暂时不关心。当我们只关心变量的部分结果时，可以使用下划线来表示不需要的部分，并节省内存。

            在这段代码中，下划线 "_" 用作临时变量，表示我们对最大值不感兴趣，只关心最大值所在的索引。因此，我们将最大值所在的索引存储在变量 "predicted" 中，而不关心最大值本身。
                    --- chatGPT
            '''
            total += labels.size(0) # 元组 [行数， 列数]
            correct += (predicted == labels).sum().item() # 张量之间的比较运算
    print("Accuracy on test set: %d %%" % (100 * correct / total))

if __name__ == "__main__":
    for epoch in range(10):
        train(epoch)
        test()

[ 1,   300] loss: 2.162
[ 1,   600] loss: 2.932
[ 1,   900] loss: 3.350
Accuracy on test set: 89 %
[ 2,   300] loss: 0.323
[ 2,   600] loss: 0.588
[ 2,   900] loss: 0.806
Accuracy on test set: 93 %
[ 3,   300] loss: 0.187
[ 3,   600] loss: 0.359
[ 3,   900] loss: 0.507
Accuracy on test set: 95 %
[ 4,   300] loss: 0.132
[ 4,   600] loss: 0.250
[ 4,   900] loss: 0.370
Accuracy on test set: 96 %
[ 5,   300] loss: 0.097
[ 5,   600] loss: 0.190
[ 5,   900] loss: 0.281
Accuracy on test set: 97 %
[ 6,   300] loss: 0.073
[ 6,   600] loss: 0.151
[ 6,   900] loss: 0.223
Accuracy on test set: 97 %
[ 7,   300] loss: 0.059
[ 7,   600] loss: 0.119
[ 7,   900] loss: 0.179
Accuracy on test set: 97 %
[ 8,   300] loss: 0.045
[ 8,   600] loss: 0.096
[ 8,   900] loss: 0.144
Accuracy on test set: 97 %
[ 9,   300] loss: 0.039
[ 9,   600] loss: 0.081
[ 9,   900] loss: 0.118
Accuracy on test set: 97 %
[ 10,   300] loss: 0.031
[ 10,   600] loss: 0.062
[ 10,   900] loss: 0.098
Accuracy on test set: 97 %
